# IMDB Movie Review Sentiment Analysis

In [1]:
import pandas as pd

In [26]:
import warnings
warnings.filterwarnings('ignore')

## Importing Dataset

In [2]:
df_reviews = pd.read_csv('IMDB-Dataset.csv')
df_reviews

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive
...,...,...
49995,I thought this movie did a down right good job...,positive
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",negative
49997,I am a Catholic taught in parochial elementary...,negative
49998,I'm going to have to disagree with the previou...,negative


## Creating Training Dataset by fixing imbalanced data (simulated through a 9:1 imbalanced dataset)

In [3]:
df_positive = df_reviews[df_reviews['sentiment'] == 'positive'][:9000]
df_negative = df_reviews[df_reviews['sentiment'] == 'negative'][:1000]

df_reviews_imb = pd.concat([df_positive, df_negative])

df_reviews_imb.value_counts('sentiment')

sentiment
positive    9000
negative    1000
Name: count, dtype: int64

In [27]:
from imblearn.under_sampling import RandomUnderSampler

rus = RandomUnderSampler(random_state = 0)
df_reviews_bal, df_reviews_bal['sentiment'] = rus.fit_resample(df_reviews_imb[['review']], df_reviews_imb['sentiment'])

## Data Splitting and Numerical Representation

In [5]:
from sklearn.model_selection import train_test_split

train, test = train_test_split(df_reviews_bal, test_size = 0.33, random_state = 42)

train_x, train_y = train['review'], train['sentiment']
test_x, test_y = test['review'], test['sentiment']

In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(stop_words = 'english')

train_x_vector = tfidf.fit_transform(train_x)

test_x_vector = tfidf.transform(test_x)

## Model Selection (Training)

In [23]:
from sklearn.svm import SVC

svc = SVC(kernel = 'linear')

svc.fit(train_x_vector, train_y)

SVC(kernel='linear')

In [10]:
print(svc.predict(tfidf.transform(['A good movie for families'])))
print(svc.predict(tfidf.transform(['Excellent pass time'])))
print(svc.predict(tfidf.transform([" I did not like this movie"])))

['positive']
['positive']
['negative']


In [11]:
from sklearn.tree import DecisionTreeClassifier

dec_tree = DecisionTreeClassifier()
dec_tree.fit(train_x_vector, train_y)

DecisionTreeClassifier()

In [13]:
from sklearn.naive_bayes import GaussianNB

gnb = GaussianNB()
gnb.fit(train_x_vector.toarray(), train_y)

GaussianNB()

## Model Evaluation

In [14]:
# calculate confusion matrix of svc
from sklearn.metrics import confusion_matrix

confusion_matrix(test_y,
                 svc.predict(test_x_vector),
                 labels=['positive', 'negative'])

array([[290,  45],
       [ 60, 265]])

In [17]:
# calculate accuracy of all models

print(svc.score(test_x_vector, test_y))
print(dec_tree.score(test_x_vector, test_y))
print(gnb.score(test_x_vector.toarray(), test_y))
print(log_reg.score(test_x_vector, test_y))

0.8409090909090909
0.646969696969697
0.6348484848484849
0.8303030303030303


In [16]:
from sklearn.metrics import f1_score

f1_score(test_y, svc.predict(test_x_vector),
         labels=['positive', 'negative'],
         average=None)

array([0.84671533, 0.83464567])

In [18]:
from sklearn.metrics import classification_report

print(classification_report(test_y,
                      svc.predict(test_x_vector),
                      labels=['positive', 'negative']))

              precision    recall  f1-score   support

    positive       0.83      0.87      0.85       335
    negative       0.85      0.82      0.83       325

    accuracy                           0.84       660
   macro avg       0.84      0.84      0.84       660
weighted avg       0.84      0.84      0.84       660



## Tuning the Model

In [19]:
from sklearn.model_selection import GridSearchCV

parameters = {'C': [1,4,8,16,32], 'kernel':['linear', 'rbf']}
svc = SVC()
svc_grid = GridSearchCV(svc, parameters, cv=5)
svc_grid.fit(train_x_vector, train_y)

GridSearchCV(cv=5, estimator=SVC(),
             param_grid={'C': [1, 4, 8, 16, 32], 'kernel': ['linear', 'rbf']})

In [20]:
best_svc = svc_grid.best_estimator_

In [21]:
grid_predictions = svc_grid.predict(test_x_vector)

In [24]:
print(classification_report(test_y, svc.predict(test_x_vector)))

              precision    recall  f1-score   support

    negative       0.85      0.82      0.83       325
    positive       0.83      0.87      0.85       335

    accuracy                           0.84       660
   macro avg       0.84      0.84      0.84       660
weighted avg       0.84      0.84      0.84       660



In [25]:
print(classification_report(test_y, grid_predictions))

              precision    recall  f1-score   support

    negative       0.85      0.82      0.83       325
    positive       0.83      0.87      0.85       335

    accuracy                           0.84       660
   macro avg       0.84      0.84      0.84       660
weighted avg       0.84      0.84      0.84       660

